# 肺叶分割模型可视化 (Dataset703_LungLobes)

基于 nnUNet 训练的 5 类肺叶分割模型，对 3 例胸部 CT 进行推理和可视化展示。

**标签:**
- 1: Left Upper Lobe (左上叶)
- 2: Left Lower Lobe (左下叶)
- 3: Right Upper Lobe (右上叶)
- 4: Right Middle Lobe (右中叶)
- 5: Right Lower Lobe (右下叶)

In [ ]:
%matplotlib inline
import sys, os, json, glob
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from pathlib import Path
import pydicom
from scipy.ndimage import zoom
from IPython.display import display as ipydisplay

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 项目根目录和 backend 路径
PROJECT_ROOT = Path(r'D:\MedSim_Studio')
DATA_DIR = PROJECT_ROOT / 'data' / 'lung_lobe_samples'
BACKEND_DIR = str(PROJECT_ROOT / 'backend')
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

print('Environment ready')

## 1. 辅助函数定义

In [ ]:
def read_dicom_series(dicom_dir):
    """读取 DICOM 系列，返回 3D volume (HU) 和 spacing (z,y,x)"""
    files = sorted(glob.glob(os.path.join(dicom_dir, '*.dcm')))
    if not files:
        raise FileNotFoundError(f'No DICOM files found in {dicom_dir}')
    slices = [pydicom.dcmread(f, force=True) for f in files]
    try:
        slices.sort(key=lambda s: float(s.InstanceNumber))
    except:
        try:
            slices.sort(key=lambda s: float(s.SliceLocation))
        except:
            pass
    pixel_data = []
    for s in slices:
        arr = s.pixel_array.astype(np.float32)
        slope = float(getattr(s, 'RescaleSlope', 1))
        intercept = float(getattr(s, 'RescaleIntercept', 0))
        arr = arr * slope + intercept
        pixel_data.append(arr)
    volume = np.stack(pixel_data, axis=0).astype(np.float32)
    volume = np.clip(volume, -1024, 3071)
    try:
        ps = tuple(map(float, slices[0].PixelSpacing))
        st = float(getattr(slices[0], 'SliceThickness', 1.0))
        spacing = (st, ps[0], ps[1])  # (z, y, x)
    except:
        spacing = (1.0, 1.0, 1.0)
    return volume, spacing, slices


def read_segmentation_dicom(dcm_path):
    """读取分割 DICOM，返回 3D binary mask (z,y,x)"""
    ds = pydicom.dcmread(dcm_path, force=True)
    if hasattr(ds, 'PerFrameFunctionalGroupsSequence') and hasattr(ds, 'pixel_array'):
        arr = ds.pixel_array
        if arr.ndim == 4:
            mask = arr[:, 0, :, :]
        else:
            mask = arr
        labels = {}
        if hasattr(ds, 'SegmentSequence'):
            for seg in ds.SegmentSequence:
                n = getattr(seg, 'SegmentNumber', 0)
                labels[n] = getattr(seg, 'SegmentLabel', str(n))
        return np.array(mask, dtype=np.uint8), labels
    try:
        return np.array(ds.pixel_array, dtype=np.uint8), {}
    except:
        return None, {}


# ─── 颜色 / 标签 / CT 窗宽窗位 ───
LUNG_LOBE_COLORS = {
    0: (0.0, 0.0, 0.0), 1: (0.39, 0.78, 0.59),
    2: (0.24, 0.63, 0.39), 3: (0.39, 0.59, 0.86),
    4: (0.51, 0.67, 0.94), 5: (0.27, 0.39, 0.78),
}
LUNG_LOBE_NAMES = {
    0: 'Background', 1: 'LUL (左上叶)', 2: 'LLL (左下叶)',
    3: 'RUL (右上叶)', 4: 'RML (右中叶)', 5: 'RLL (右下叶)',
}
CT_WW, CT_WL = 1500, -600  # 肺窗


def window_ct(vol, ww=CT_WW, wl=CT_WL):
    lo, hi = wl - ww/2, wl + ww/2
    return np.clip((vol - lo) / (hi - lo), 0, 1)


def overlay_mask(ct_slice, mask_slice, alpha=0.5):
    c = window_ct(ct_slice)
    rgb = np.stack([c]*3, axis=-1)
    ov = np.zeros((*ct_slice.shape, 4))
    for lid, col in LUNG_LOBE_COLORS.items():
        if lid == 0:
            continue
        m = mask_slice == lid
        for ch in range(3):
            ov[:,:,ch][m] = col[ch]
        ov[:,:,3][m] = alpha
    return rgb, ov


def lobe_cmap():
    return mcolors.ListedColormap([LUNG_LOBE_COLORS[i] for i in range(6)])


def legend_handles():
    return [Patch(color=LUNG_LOBE_COLORS[i], label=LUNG_LOBE_NAMES[i]) for i in range(1, 6)]


def show_fig(fig):
    """显示图片并释放内存（不调用 plt.show() 以避免 warning）"""
    fig.canvas.draw()
    ipydisplay(fig)
    plt.close(fig)


print('Helper functions defined')

## 2. 加载模型

In [ ]:
import torch
from app.core.config import settings
from app.ai.nnunet_lung_lobe import run_nnunet_lung_lobe, is_available

LOCAL_MODEL_PATH = r'D:\MedSim_Studio\models\nnunet_lung_lobe'
settings.NNUNET_LUNG_LOBE_MODEL_PATH = LOCAL_MODEL_PATH
print(f'Model available: {is_available()}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

## 3. 读取并处理各病例

In [ ]:
cases = []
for case_dir in sorted(DATA_DIR.iterdir()):
    if not case_dir.is_dir():
        continue
    ct_dir, seg_file = None, None
    for item in case_dir.rglob('*'):
        if item.is_dir() and 'Segmentation' not in item.name:
            dcm = list(item.glob('*.dcm'))
            if len(dcm) > 20:
                ct_dir = item
        elif item.is_file() and 'Segmentation' in str(item) and item.suffix == '.dcm':
            seg_file = item
    if ct_dir:
        cases.append({'name': case_dir.name, 'ct_dir': ct_dir, 'seg_file': seg_file})
        print(f'{case_dir.name}: CT={ct_dir.name} ({len(list(ct_dir.glob("*.dcm")))} slc)',
              f'Seg={seg_file.name if seg_file else "N/A"}')
print(f'\nTotal: {len(cases)} cases')

In [ ]:
from scipy.ndimage import zoom as ndzoom

for case in cases:
    print(f'\n=== {case["name"]} ===')
    vol, sp, _ = read_dicom_series(str(case['ct_dir']))
    case['volume'] = vol; case['spacing'] = sp
    print(f'  Volume: {vol.shape}, HU: [{vol.min():.0f}, {vol.max():.0f}]')

    if case['seg_file']:
        sm, _ = read_segmentation_dicom(str(case['seg_file']))
        case['seg_mask'] = sm
        print(f'  Seg: {sm.shape if sm is not None else "None"}, unique={np.unique(sm) if sm is not None else "N/A"}')
    else:
        case['seg_mask'] = None

    # 降采样 → 1.5mm 各向同性
    target = 1.5
    factors = (sp[0]/target, sp[1]/target, sp[2]/target)
    vd = ndzoom(vol.astype(np.float32), factors, order=1)
    if any(s < 64 for s in vd.shape):
        vd = np.pad(vd, [(0, max(0,64-s)) for s in vd.shape], mode='constant', constant_values=-1024)
    print(f'  Down: {vol.shape} -> {vd.shape}')

    print('  Running nnUNet...')
    pred = run_nnunet_lung_lobe(vd, (target, target, target))
    case['prediction_raw'] = pred
    print(f'  Pred: {pred.shape}, labels={np.unique(pred)}')

    # 上采样回原分辨率
    if pred.shape != vol.shape[:3]:
        pc = pred[:vd.shape[0], :vd.shape[1], :vd.shape[2]]
        upf = (vol.shape[0]/pc.shape[0], vol.shape[1]/pc.shape[1], vol.shape[2]/pc.shape[2])
        pu = np.round(ndzoom(pc.astype(np.float32), upf, order=0)).astype(np.int32)
        case['prediction'] = pu
        print(f'  Up: -> {pu.shape}')
    else:
        case['prediction'] = pred

## 4. 检查分割 DICOM 内容

In [ ]:
for case in cases:
    if case['seg_file']:
        ds = pydicom.dcmread(str(case['seg_file']), force=True)
        print(f'\n=== {case["name"]} - Segmentation Info ===')
        for attr in ['SOPClassUID', 'Modality', 'NumberOfFrames', 'Rows', 'Columns']:
            if hasattr(ds, attr):
                print(f'  {attr}: {getattr(ds, attr)}')
        if hasattr(ds, 'SegmentSequence'):
            for seg in ds.SegmentSequence:
                print(f'  Segment {getattr(seg,"SegmentNumber","?")}: {getattr(seg,"SegmentLabel","?")}')
        print(f'  pixel_array: {ds.pixel_array.shape}, unique={np.unique(ds.pixel_array)}')

## 5. 单病例全方位展示 (轴位/冠状位/矢状位)

5 列：CT / GT / Pred / CT+GT / CT+Pred

In [ ]:
def plot_triplanar(case):
    vol, pred, seg = case['volume'], case['prediction'], case.get('seg_mask')
    z, y, x = vol.shape
    # 找肺区中心
    zi = np.where(np.any(pred>0, axis=(1,2)))[0]; z_mid = zi[len(zi)//2] if len(zi)>0 else z//2
    yi = np.where(np.any(pred>0, axis=(0,2)))[0]; y_mid = yi[len(yi)//2] if len(yi)>0 else y//2
    xi = np.where(np.any(pred>0, axis=(0,1)))[0]; x_mid = xi[len(xi)//2] if len(xi)>0 else x//2

    views = [(vol[z_mid], pred[z_mid], 'Axial'),
             (vol[:,y_mid,:], pred[:,y_mid,:], 'Coronal'),
             (vol[:,:,x_mid], pred[:,:,x_mid], 'Sagittal')]
    seg_ok = seg is not None and seg.shape == vol.shape
    cmap = lobe_cmap()

    fig, axes = plt.subplots(3, 5, figsize=(20, 12))
    for r, (ct_s, pr_s, name) in enumerate(views):
        gt_s = (seg[r*len(seg)//3] if seg_ok else None) if seg_ok else None
        # col 0: CT
        axes[r,0].imshow(window_ct(ct_s), cmap='gray'); axes[r,0].set_title(f'{name} - CT'); axes[r,0].axis('off')
        # col 1: GT
        if seg_ok and seg.max()>0:
            axes[r,1].imshow(seg_int, cmap=cmap, vmin=0, vmax=5, interpolation='nearest')
            axes[r,1].set_title(f'{name} - GT')
        else:
            axes[r,1].text(.5,.5,'No GT',ha='center',va='center',transform=axes[r,1].transAxes)
            axes[r,1].set_title(f'{name} - GT')
        axes[r,1].axis('off')
        # col 2: Pred
        axes[r,2].imshow(pr_s, cmap=cmap, vmin=0, vmax=5, interpolation='nearest')
        axes[r,2].set_title(f'{name} - Pred'); axes[r,2].axis('off')
        # col 3: CT+GT
        if seg_ok and seg.max()>0:
            c3, o3 = overlay_mask(ct_s, seg_int, 0.4)
            axes[r,3].imshow(c3); axes[r,3].imshow(o3)
            axes[r,3].set_title(f'{name} - CT+GT')
        else:
            axes[r,3].text(.5,.5,'No GT',ha='center',va='center',transform=axes[r,3].transAxes)
            axes[r,3].set_title(f'{name} - CT+GT')
        axes[r,3].axis('off')
        # col 4: CT+Pred
        c4, o4 = overlay_mask(ct_s, pr_s, 0.4)
        axes[r,4].imshow(c4); axes[r,4].imshow(o4)
        axes[r,4].set_title(f'{name} - CT+Pred'); axes[r,4].axis('off')

    fig.legend(handles=legend_handles(), loc='lower center', ncol=5, bbox_to_anchor=(0.5,-0.02))
    plt.suptitle(f'Case: {case["name"]} — 三切面对比', fontsize=16, y=1.01)
    plt.tight_layout(); show_fig(fig)


if cases:
    plot_triplanar(cases[0])

## 6. 多 slice 连续展示 (轴位)

In [ ]:
def plot_slice_grid(case, n_cols=4, n_slices=8):
    vol, pred = case['volume'], case['prediction']
    zi = np.where(np.any(pred>0, axis=(1,2)))[0]
    if len(zi) > n_slices:
        idx = zi[::len(zi)//n_slices][:n_slices]
    elif len(zi) > 0:
        idx = zi
    else:
        idx = np.linspace(0, vol.shape[0]-1, n_slices, dtype=int)

    nr = int(np.ceil(len(idx)/n_cols))
    fig, axes = plt.subplots(nr, n_cols*2, figsize=(4*n_cols, 4*nr))
    if nr == 1:
        axes = axes.reshape(1, -1)

    for i, si in enumerate(idx):
        r, c = i//n_cols, (i%n_cols)*2
        axes[r,c].imshow(window_ct(vol[si]), cmap='gray')
        axes[r,c].set_title(f'Sl {si} - CT'); axes[r,c].axis('off')
        ct_rgb, ov = overlay_mask(vol[si], pred[si], 0.4)
        axes[r,c+1].imshow(ct_rgb); axes[r,c+1].imshow(ov)
        axes[r,c+1].set_title(f'Sl {si} - CT+Pred'); axes[r,c+1].axis('off')
    for i in range(len(idx), nr*n_cols):
        r, c = i//n_cols, (i%n_cols)*2
        axes[r,c].axis('off'); axes[r,c+1].axis('off')
    plt.suptitle(f'{case["name"]} — 连续切片 (轴位)', fontsize=14)
    plt.tight_layout(); show_fig(fig)


for case in cases:
    plot_slice_grid(case)

## 7. 预测轮廓叠加

In [ ]:
from skimage import measure

def plot_contours(case):
    vol, pred = case['volume'], case['prediction']
    zi = np.where(np.any(pred>0, axis=(1,2)))[0]
    # chose 4 representative slices
    if len(zi) >= 4:
        si = [zi[len(zi)//5*1], zi[len(zi)//5*2], zi[len(zi)//5*3], zi[len(zi)//5*4]]
    else:
        si = [vol.shape[0]//3, vol.shape[0]//2, 2*vol.shape[0]//3]

    fig, axes = plt.subplots(2, len(si), figsize=(6*len(si), 10))
    for i, s in enumerate(si):
        ct_s, pr_s = vol[s], pred[s]
        ax1, ax2 = axes[0,i] if len(si)>1 else axes[0], axes[1,i] if len(si)>1 else axes[1]
        ax1.imshow(window_ct(ct_s), cmap='gray')
        for lid in range(1,6):
            m = pr_s == lid
            if m.sum() < 10:
                continue
            try:
                for c in measure.find_contours(m, 0.5):
                    ax1.plot(c[:,1], c[:,0], color=LUNG_LOBE_COLORS[lid], lw=1.5)
            except:
                pass
        ax1.set_title(f'Sl {s} - 轮廓'); ax1.axis('off')
        ct_rgb, ov = overlay_mask(ct_s, pr_s, 0.4)
        ax2.imshow(ct_rgb); ax2.imshow(ov)
        for lid in range(1,6):
            m = pr_s == lid
            if m.sum() < 10:
                continue
            try:
                for c in measure.find_contours(m, 0.5):
                    ax2.plot(c[:,1], c[:,0], color='white', lw=1)
            except:
                pass
        ax2.set_title(f'Sl {s} - 填充+轮廓'); ax2.axis('off')
    plt.suptitle(f'{case["name"]}', fontsize=14)
    plt.tight_layout(); show_fig(fig)


for case in cases:
    plot_contours(case)

## 8. GT vs 预测对比图

Segmentation DICOM 为 binary lung mask (0/1)，预测为 5 类肺叶。
将预测合并为 binary 后对比。

In [ ]:
def plot_comparison(case):
    vol, pred, seg = case['volume'], case['prediction'], case.get('seg_mask')
    if seg is None or seg.max() == 0:
        print(f'{case["name"]}: No GT'); return

    # resize seg to match vol
    f = (vol.shape[0]/seg.shape[0], vol.shape[1]/seg.shape[1], vol.shape[2]/seg.shape[2])
    seg_r = np.round(ndzoom(seg.astype(np.float32), f, order=0)).astype(np.uint8)
    gt = (seg_r > 0).astype(np.uint8)
    pl = (pred > 0).astype(np.uint8)

    # find slices with max diff
    dz = (pl!=gt).sum(axis=(1,2)); z_mid = np.argmax(dz)
    dy = (pl!=gt).sum(axis=(0,2)); y_mid = np.argmax(dy)
    dx = (pl!=gt).sum(axis=(0,1)); x_mid = np.argmax(dx)

    views = [(vol[z_mid], pred[z_mid], gt[z_mid], pl[z_mid], f'Axial z={z_mid}'),
             (vol[:,y_mid,:], pred[:,y_mid,:], gt[:,y_mid,:], pl[:,y_mid,:], f'Coronal y={y_mid}'),
             (vol[:,:,x_mid], pred[:,:,x_mid], gt[:,:,x_mid], pl[:,:,x_mid], f'Sagittal x={x_mid}')]

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for r, (ct_s, pr_s, gt_s, pl_s, tt) in enumerate(views):
        ct_n = window_ct(ct_s); ct_r = np.stack([ct_n]*3, axis=-1)
        # CT+GT (red)
        ov_gt = np.zeros((*ct_s.shape,4)); ov_gt[:,:,0] = gt_s*0.5; ov_gt[:,:,3] = gt_s*0.4
        axes[r,0].imshow(ct_r); axes[r,0].imshow(ov_gt)
        axes[r,0].set_title(f'{tt} - CT+GT'); axes[r,0].axis('off')
        # CT+Pred
        c2, o2 = overlay_mask(ct_s, pr_s, 0.4)
        axes[r,1].imshow(c2); axes[r,1].imshow(o2)
        axes[r,1].set_title(f'{tt} - CT+Pred'); axes[r,1].axis('off')
        # Diff
        diff = np.zeros_like(gt_s, dtype=float); fp=pl_s&~gt_s; fn=~pl_s&gt_s; tp=pl_s&gt_s
        diff[tp]=0.5; diff[fp]=0.8; diff[fn]=0.2
        axes[r,2].imshow(ct_n, cmap='gray', alpha=0.5)
        axes[r,2].imshow(diff, cmap='RdYlBu_r', vmin=0, vmax=1, alpha=0.7)
        axes[r,2].set_title(f'{tt} - TP/FP/FN'); axes[r,2].axis('off')
        # Overlap
        axes[r,3].imshow(ct_n, cmap='gray', alpha=0.5)
        axes[r,3].imshow(diff, cmap='RdYlBu_r', vmin=0, vmax=1, alpha=0.6)
        axes[r,3].set_title(f'{tt} - 重叠分析'); axes[r,3].axis('off')
    plt.suptitle(f'{case["name"]} — Binary GT vs Pred', fontsize=14)
    plt.tight_layout(); show_fig(fig)


for case in cases:
    plot_comparison(case)

## 9. Dice 统计

In [ ]:
def compute_dice(pred, gt_bin):
    pl = (pred>0).astype(np.uint8); gt = (gt_bin>0).astype(np.uint8)
    inter = (pl&gt).sum(); ps = pl.sum(); gs = gt.sum()
    lung_dice = 2*inter/(ps+gs) if ps+gs>0 else float('nan')
    lobe = {}
    for lid in range(1,6):
        pm = pred==lid; i2 = (pm&gt).sum(); p2 = pm.sum()
        lobe[lid] = 2*i2/(p2+gs) if p2+gs>0 else float('nan')
    return lung_dice, lobe


all_dice = {}
for case in cases:
    seg = case.get('seg_mask')
    if seg is None or seg.max()==0:
        continue
    pred = case['prediction']
    f = (pred.shape[0]/seg.shape[0], pred.shape[1]/seg.shape[1], pred.shape[2]/seg.shape[2])
    seg_r = np.round(ndzoom(seg.astype(np.float32), f, order=0)).astype(np.uint8)
    ld, lb = compute_dice(pred, seg_r)
    all_dice[case['name']] = (ld, lb)
    print(f'\n{case["name"]} — Lung Dice: {ld:.4f}')
    for lid in range(1,6):
        d = lb.get(lid, float('nan'))
        print(f'  {LUNG_LOBE_NAMES[lid]}: {"N/A" if np.isnan(d) else f"{d:.4f}"}')

if all_dice:
    names = list(all_dice.keys())
    fig, axes = plt.subplots(1, 2, figsize=(14,5))
    # binary
    vals = [all_dice[n][0] for n in names]
    axes[0].bar(names, vals, color=['#2196F3','#FF9800','#4CAF50'], alpha=0.8)
    for b,v in zip(axes[0].patches, vals):
        axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{v:.4f}', ha='center', va='bottom')
    axes[0].set_ylabel('Dice'); axes[0].set_ylim([0,1.15])
    axes[0].axhline(y=0.8, c='gray', ls='--'); axes[0].set_title('Binary Lung Dice')
    # per-lobe
    x = np.arange(5); w = 0.8/len(names)
    for i, n in enumerate(names):
        s = [all_dice[n][1].get(lid,0) or 0 for lid in range(1,6)]
        axes[1].bar(x+(i-(len(names)-1)/2)*w, s, w, label=n, alpha=0.8)
    axes[1].set_xticks(x); axes[1].set_xticklabels(['LUL','LLL','RUL','RML','RLL'])
    axes[1].set_ylabel('Dice vs GT lung'); axes[1].set_ylim([0,1.15]); axes[1].legend()
    axes[1].axhline(y=0.8, c='gray', ls='--')
    plt.suptitle('Dice Similarity (Binary Lung GT)')
    plt.tight_layout(); show_fig(fig)

## 10. 各病例详细展示

In [ ]:
def plot_case_detail(case):
    vol, pred = case['volume'], case['prediction']
    zi = np.where(np.any(pred>0, axis=(1,2)))[0]
    if len(zi) >= 4:
        si = [zi[len(zi)//4*1], zi[len(zi)//4*2], zi[len(zi)//4*3]]
    else:
        si = [vol.shape[0]//3, vol.shape[0]//2, 2*vol.shape[0]//3]
    cmap = lobe_cmap()
    fig, axes = plt.subplots(3, 4, figsize=(16,12))
    for i, s in enumerate(si):
        ct_s, pr_s = vol[s], pred[s]
        axes[i,0].imshow(window_ct(ct_s), cmap='gray')
        axes[i,0].set_title(f'CT sl {s}'); axes[i,0].axis('off')
        axes[i,1].imshow(pr_s, cmap=cmap, vmin=0, vmax=5, interpolation='nearest')
        axes[i,1].set_title(f'Pred sl {s}'); axes[i,1].axis('off')
        c3, o3 = overlay_mask(ct_s, pr_s, 0.4)
        axes[i,2].imshow(c3); axes[i,2].imshow(o3)
        axes[i,2].set_title(f'CT+Pred sl {s}'); axes[i,2].axis('off')
        axes[i,3].imshow(window_ct(ct_s), cmap='gray')
        for lid in range(1,6):
            m = pr_s == lid
            if m.sum() < 10: continue
            try:
                for c in measure.find_contours(m, 0.5):
                    axes[i,3].plot(c[:,1], c[:,0], color=LUNG_LOBE_COLORS[lid], lw=2)
            except: pass
        axes[i,3].set_title(f'Contours sl {s}'); axes[i,3].axis('off')
    plt.suptitle(f'{case["name"]} | {vol.shape}', fontsize=14)
    plt.tight_layout(); show_fig(fig)


for case in cases:
    plot_case_detail(case)

## 11. 3D 表面重建 (Marching Cubes)

In [ ]:
from skimage import measure as skm
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def plot_3d(case, step=3):
    pred, sp = case['prediction'], case['spacing']
    fig = plt.figure(figsize=(14,8))
    ax1 = fig.add_subplot(121, projection='3d'); ax2 = fig.add_subplot(122, projection='3d')
    for ax, lids, title in [(ax1, [1,2], 'Left'), (ax2, [3,4,5], 'Right')]:
        for lid in lids:
            m = pred == lid
            if m.sum() < 100: continue
            md = m[::step, ::step, ::step]
            try:
                v, f, _, _ = skm.marching_cubes(md, 0.5)
                vs = v * step
                vs[:,0] *= sp[2]; vs[:,1] *= sp[1]; vs[:,2] *= sp[0]
                ax.add_collection3d(Poly3DCollection(vs[f], alpha=0.7,
                    facecolor=LUNG_LOBE_COLORS[lid], edgecolor='none'))
            except Exception as e:
                print(f'  {LUNG_LOBE_NAMES[lid]}: {e}')
        ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z'); ax.set_title(title)
    plt.suptitle(f'{case["name"]} — 3D Lung Lobes', fontsize=14)
    plt.tight_layout(); show_fig(fig)


for case in cases:
    print(f'\n=== {case["name"]} 3D ===')
    plot_3d(case)

## 12. 冠状位连续切片

In [ ]:
def plot_coronal(case, n=6):
    vol, pred = case['volume'], case['prediction']
    yi = np.where(np.any(pred>0, axis=(0,2)))[0]
    if len(yi) > n:
        yi = yi[::len(yi)//n][:n]
    cols = min(n,6); rows = int(np.ceil(n/cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols,4*rows))
    axes = axes.flatten()
    for i, y in enumerate(yi):
        c, o = overlay_mask(vol[:,y,:], pred[:,y,:], 0.4)
        axes[i].imshow(c); axes[i].imshow(o); axes[i].set_title(f'y={y}'); axes[i].axis('off')
    for i in range(len(yi), len(axes)):
        axes[i].axis('off')
    plt.suptitle(f'{case["name"]} — Coronal', fontsize=14)
    plt.tight_layout(); show_fig(fig)


for case in cases:
    plot_coronal(case)

## 13. 矢状位连续切片

In [ ]:
def plot_sagittal(case, n=6):
    vol, pred = case['volume'], case['prediction']
    xi = np.where(np.any(pred>0, axis=(0,1)))[0]
    if len(xi) > n:
        xi = xi[::len(xi)//n][:n]
    cols = min(n,6); rows = int(np.ceil(n/cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols,4*rows))
    axes = axes.flatten()
    for i, x in enumerate(xi):
        c, o = overlay_mask(vol[:,:,x], pred[:,:,x], 0.4)
        axes[i].imshow(c); axes[i].imshow(o); axes[i].set_title(f'x={x}'); axes[i].axis('off')
    for i in range(len(xi), len(axes)):
        axes[i].axis('off')
    plt.suptitle(f'{case["name"]} — Sagittal', fontsize=14)
    plt.tight_layout(); show_fig(fig)


for case in cases:
    plot_sagittal(case)

## 14. 肺叶体积统计

In [ ]:
def lobe_volume(mask, sp):
    vox_ml = sp[0]*sp[1]*sp[2]/1000
    return {lid: float((mask==lid).sum()*vox_ml) for lid in range(1,6)}

fig, axes = plt.subplots(1, len(cases), figsize=(6*len(cases),5))
if len(cases) == 1:
    axes = [axes]
for i, case in enumerate(cases):
    vol = lobe_volume(case['prediction'], case['spacing'])
    lbls = [LUNG_LOBE_NAMES[lid] for lid in range(1,6)]
    vals = [vol[lid]/1000 for lid in range(1,6)]
    cols = [LUNG_LOBE_COLORS[lid] for lid in range(1,6)]
    bars = axes[i].barh(lbls, vals, color=cols)
    for b, v in zip(bars, vals):
        axes[i].text(b.get_width()+0.01, b.get_y()+b.get_height()/2, f'{v:.2f}L', va='center')
    axes[i].set_xlabel('Vol (L)'); axes[i].set_title(f'{case["name"]}')
plt.tight_layout(); show_fig(fig)

print('\nLobe Volume Statistics:')
print(f'{"Case":<12}{"Lobe":<18}{"Vol (ml)":<12}{"%"}')
print('-'*55)
for case in cases:
    vol = lobe_volume(case['prediction'], case['spacing'])
    total = sum(vol.values())
    for lid in range(1,6):
        v = vol[lid]; print(f'{case["name"]:<12}{LUNG_LOBE_NAMES[lid]:<18}{v:<12.1f}{v/total*100:<6.1f}%')
    print(f'{"":<12}{"Total":<18}{total:<12.1f}100.0%\n')

## 15. 精度评估 (Binary Lung Mask vs 5-class Prediction)

In [ ]:
print('Binary Lung Segmentation Evaluation')
print('='*65)
for case in cases:
    seg = case.get('seg_mask')
    if seg is None or seg.max()==0:
        print(f'{case["name"]}: No GT'); continue
    pred = case['prediction']
    f = (pred.shape[0]/seg.shape[0], pred.shape[1]/seg.shape[1], pred.shape[2]/seg.shape[2])
    seg_r = np.round(ndzoom(seg.astype(np.float32), f, order=0)).astype(np.uint8)
    gt = (seg_r>0).astype(bool); pl = (pred>0).astype(bool)
    tp=(pl&gt).sum(); fp=(pl&~gt).sum(); fn=(~pl&gt).sum(); tn=(~pl&~gt).sum()
    prec=tp/(tp+fp) if tp+fp>0 else float('nan')
    rec=tp/(tp+fn) if tp+fn>0 else float('nan')
    acc=(tp+tn)/(tp+tn+fp+fn) if tp+tn+fp+fn>0 else float('nan')
    iou=tp/(tp+fp+fn) if tp+fp+fn>0 else float('nan')
    dice=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn>0 else float('nan')
    print(f'\n{case["name"]}: TP={tp:,} FP={fp:,} FN={fn:,} TN={tn:,}')
    print(f'  Prec={prec:.4f} Rec={rec:.4f} Acc={acc:.4f} IoU={iou:.4f} Dice={dice:.4f}')
    total_lung_vox = tp+fn
    for lid in range(1,6):
        pct = (pred==lid).sum()/total_lung_vox*100 if total_lung_vox>0 else 0
        print(f'  {LUNG_LOBE_NAMES[lid]}: {pct:.1f}% of GT lung')

## 16. 综合 3×3 网格：3 病例 × 3 切面

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15,15))
for ci, case in enumerate(cases):
    vol, pred = case['volume'], case['prediction']
    zi = np.where(np.any(pred>0, axis=(1,2)))[0]; zm = zi[len(zi)//2] if len(zi)>0 else vol.shape[0]//2
    yi = np.where(np.any(pred>0, axis=(0,2)))[0]; ym = yi[len(yi)//2] if len(yi)>0 else vol.shape[1]//2
    xi = np.where(np.any(pred>0, axis=(0,1)))[0]; xm = xi[len(xi)//2] if len(xi)>0 else vol.shape[2]//2
    for ri, (sl, name) in enumerate([(vol[zm],'Axial'), (vol[:,ym,:],'Coronal'), (vol[:,:,xm],'Sagittal')]):
        c, o = overlay_mask(sl, [pred[zm], pred[:,ym,:], pred[:,:,xm]][ri], 0.4)
        axes[ri,ci].imshow(c); axes[ri,ci].imshow(o)
        if ri==0: axes[ri,ci].set_title(case['name'], fontsize=14, fontweight='bold')
        if ci==0: axes[ri,ci].set_ylabel(name, fontsize=12, fontweight='bold')
        axes[ri,ci].axis('off')
plt.suptitle('3-Case × 3-View Comparison', fontsize=16, y=0.98)
plt.tight_layout(); show_fig(fig)

---
**可视化完成.** 包含：
1. 原始 CT 图像 (轴位/冠状位/矢状位, 肺窗)  
2. Ground Truth 分割  
3. 模型 5 类肺叶预测  
4. CT + GT / CT + Pred 叠加  
5. GT vs Pred 对比 (差异图 + 重叠分析)  
6. 3 个病例各自详细展示  
7. 多 slice 连续展示 (3 个切面)  
8. 预测轮廓叠加  
9. 3D 表面重建  
10. Dice 系数 + 肺叶体积统计  
11. 精度评估 (Precision/Recall/IoU)  
12. 3×3 综合对比网格